# PCD Assignment 02 — Image Enhancement

**Kasus:** Blurred image  
**Metode:** Spatial filtering dengan convolution.

- Gaussian filter → membuat/memodelkan blur.
- Sharpening → memperkuat edge.
- Unsharp masking → menambahkan kembali detail yang hilang.

\[
g(x,y)=\sum_i\sum_j w(i,j)f(x-i,y-j)
\]

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

img = cv2.cvtColor(cv2.imread('/Users/rpurusa/Downloads/pcd_02/pcd_02.webp', cv2.COLOR_BGR2RGB).astype(np.float32)

def conv(img, kernel, clip=True):
    kh, kw = kernel.shape
    ph, pw = kh // 2, kw // 2
    pad = np.pad(img, ((ph, ph), (pw, pw), (0, 0)), mode="edge")
    out = np.zeros_like(img)

    for i in range(kh):
        for j in range(kw):
            out += kernel[i, j] * pad[i:i+img.shape[0], j:j+img.shape[1]]

    return np.clip(out, 0, 255) if clip else out

gauss = np.array([
    [1,  4,  6,  4, 1],
    [4, 16, 24, 16, 4],
    [6, 24, 36, 24, 6],
    [4, 16, 24, 16, 4],
    [1,  4,  6,  4, 1]
], np.float32)
gauss /= gauss.sum()

blur = conv(img, gauss)

: 

In [ ]:
sharp_kernel = np.array([
    [0, -1,  0],
    [-1, 5, -1],
    [0, -1,  0]
], np.float32)

sharp = conv(blur, sharp_kernel)

smooth = conv(blur, gauss)
detail = blur - smooth
unsharp = np.clip(blur + 2 * detail, 0, 255)

results = {
    "Original": img,
    "Blurred": blur,
    "Sharpen": sharp,
    "Unsharp": unsharp
}

for name, im in results.items():
    cv2.imwrite(
        str(IMG / f"{name.lower()}.png"),
        cv2.cvtColor(im.astype(np.uint8), cv2.COLOR_RGB2BGR)
    )

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, (name, im) in zip(ax, results.items()):
    a.imshow(im.astype(np.uint8))
    a.set_title(name)
    a.axis("off")
plt.tight_layout()

In [ ]:
def psnr(ref, im):
    mse = np.mean((ref - im) ** 2)
    return 20 * np.log10(255 / np.sqrt(mse))

lap = np.array([[0, 1, 0],
                [1,-4, 1],
                [0, 1, 0]], np.float32)

print(f"{'Image':<10} {'PSNR':>8} {'Sharpness':>12}")
for name, im in {"Blurred": blur, "Sharpen": sharp, "Unsharp": unsharp}.items():
    gray = im.mean(axis=2, keepdims=True)
    score = conv(gray, lap, clip=False).var()
    print(f"{name:<10} {psnr(img, im):>8.2f} {score:>12.2f}")

## Analisis

Gaussian filter bekerja sebagai **low-pass filter**, sehingga detail dan edge berkurang dan gambar menjadi blur.

Sharpening menggunakan kernel convolution untuk memperbesar perubahan intensitas di sekitar edge.  
Unsharp masking menghitung `detail = image - smooth`, lalu menambahkan detail tersebut kembali ke gambar.

Hasil dibandingkan menggunakan:
- **PSNR**: semakin tinggi, semakin dekat dengan gambar referensi.
- **Sharpness**: semakin tinggi menunjukkan edge lebih kuat.

Kesimpulan: spatial filtering dengan sharpening dan unsharp masking dapat meningkatkan ketajaman gambar blur secara sederhana dan sesuai konsep convolution pada materi.